<a href="https://colab.research.google.com/github/Nazarick0912/CPC353/blob/main/CPC353_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter, defaultdict

DATA_PATH = Path(
    "/content/CPC353_G04_Final_Review_Corpus.csv"
)

OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    keep_default_na=False
)

required_columns = [
    "Review ID", "Data Origin", "Product Category",
    "Product / Eatery Name", "Source Platform", "Source URL",
    "Rating (1-5)", "Review Date", "Review Text",
    "Language Classification", "Code-Switching Type"
]

assert df.columns.tolist() == required_columns, \
    "The CSV columns do not match the agreed template."

# Preserve a copy for later checks.
original_text = df["Review Text"].copy()

print("Loaded:", DATA_PATH.name)
print("Reviews:", len(df))
display(df.head())

# Check every column for missing, empty, or whitespace-only values.
# This cell inspects the dataset without changing it.

missing_counts = df.apply(
    lambda column: (
        column.isna()
        | column.astype(str).str.strip().eq("")
    ).sum()
)

display(missing_counts.rename("Missing or empty values").to_frame())

print("Total missing or empty values:", missing_counts.sum())

Loaded: CPC353_G04_Final_Review_Corpus.csv
Reviews: 200


,Review ID,Data Origin,Product Category,Product / Eatery Name,Source Platform,Source URL,Rating (1-5),Review Date,Review Text,Language Classification,Code-Switching Type
0,REV-001,Self-Collected (Real),Ayam Gepuk,"Ayam Gepuk Original Sungai Dua, Penang",Google Reviews,https://maps.app.goo.gl/GXqZMRFTr2C68gCJ8,1,6 months ago,I’m quite disappointed with the food quality. ...,Code-switched,Intra-sentential
1,REV-002,Self-Collected (Real),Ayam Gepuk,"Ayam Gepuk Original Sungai Dua, Penang",Google Reviews,https://maps.app.goo.gl/5sXxLPesLn6jzj7L8,1,1 year ago,srsla kubis korng bg mcmni je???? I pay for 13...,Code-switched,Intra-sentential; Inter-sentential
2,REV-003,Self-Collected (Real),Ayam Gepuk,"Ayam Gepuk Original Sungai Dua, Penang",Google Reviews,https://share.google/W7iraTWGCIcoWDDYv,1,1 year ago,"Hmmmm judging from the picture, you can say it...",English,N/A
3,REV-004,Self-Collected (Real),Ayam Gepuk,"Ayam Gepuk Original Sungai Dua, Penang",Google Reviews,https://share.google/0QgUt75bNX9fwYpEV,1,1 year ago,I tak suka complain about food. But i bought t...,Code-switched,Intra-sentential; Inter-sentential
4,REV-005,Self-Collected (Real),Ayam Gepuk,"Ayam Gepuk Original Sungai Dua, Penang",Google Reviews,https://maps.app.goo.gl/skfuAanD57teeLVF9,2,1 year ago,Foods okay but lousy customer service i feel. ...,Code-switched,Intra-sentential; Inter-sentential


,Missing or empty values
Review ID,0
Data Origin,0
Product Category,0
Product / Eatery Name,0
Source Platform,0
Source URL,0
Rating (1-5),0
Review Date,0
Review Text,0
Language Classification,0


Total missing or empty values: 0


In [7]:
# Check whether any Review ID appears more than once.
# Display conflicting records without deleting or changing them.

duplicate_ids = df[
    df["Review ID"].duplicated(keep=False)
]

print("Total records:", len(df))
print("Unique Review IDs:", df["Review ID"].nunique())
print("Records with repeated IDs:", len(duplicate_ids))

if not duplicate_ids.empty:
    display(duplicate_ids)

Total records: 200
Unique Review IDs: 200
Records with repeated IDs: 0


In [8]:
# Identify records sharing the same source URL.
# Retain them all; a repeated URL does not by itself prove duplication.

url_conflicts = df[
    df["Source URL"].duplicated(keep=False)
].copy()

print("Records sharing a source URL:", len(url_conflicts))

display(
    url_conflicts[
        ["Review ID", "Source URL", "Review Text",
         "Language Classification"]
    ].sort_values(["Source URL", "Review ID"])
)

Records sharing a source URL: 4


,Review ID,Source URL,Review Text,Language Classification
53,REV-054,https://maps.app.goo.gl/cADE4T8krfMRijZB7,Semua makanan sedap okay. Smbal pedas dan lema...,Malay
63,REV-064,https://maps.app.goo.gl/cADE4T8krfMRijZB7,semua okay makanan sedap memang datang makan e...,Code-switched
65,REV-066,https://maps.app.goo.gl/dsSAJGTzU7PQWyUR8,"makanan memang sedap, service laju dan teratur.",Code-switched
77,REV-078,https://maps.app.goo.gl/dsSAJGTzU7PQWyUR8,service laju dan makanan sedap 👍🏻 harga berpat...,Code-switched


In [9]:
# Flag identical review text for inspection.
# Matching text alone does not prove that two records are duplicates.

repeated_text = df[
    df["Review Text"].duplicated(keep=False)
].copy()

print("Records sharing identical review text:", len(repeated_text))

display(
    repeated_text[
        ["Review ID", "Review Text", "Source URL"]
    ].sort_values(["Review Text", "Review ID"])
)

Records sharing identical review text: 6


,Review ID,Review Text,Source URL
145,REV-146,Sedap,https://maps.app.goo.gl/F8No1Qfg9UyjEuWLA
148,REV-149,Sedap,https://maps.app.goo.gl/fRtFw7PjbzmzMMx86
150,REV-151,Sedap,https://maps.app.goo.gl/VMbCeShpsnNAGPC39
175,REV-176,Sedap,https://maps.app.goo.gl/4GLkHh2JXk8YiSwA8
121,REV-122,sedap,https://maps.app.goo.gl/7BgUDZBUmAtwyyZD9
158,REV-159,sedap,https://maps.app.goo.gl/f4SXtwt9ZpnSiE9o7


In [10]:
# Check individual review ratings without changing the original column.
# Flag missing, nonnumeric, fractional, or out-of-range ratings.

ratings_numeric = pd.to_numeric(
    df["Rating (1-5)"],
    errors="coerce"
)

invalid_ratings = df[
    ~ratings_numeric.isin([1, 2, 3, 4, 5])
]

print("Records with invalid ratings:", len(invalid_ratings))

if not invalid_ratings.empty:
    display(invalid_ratings[["Review ID", "Rating (1-5)"]])
else:
    print("All ratings are valid whole numbers from 1 to 5.")

Records with invalid ratings: 0
All ratings are valid whole numbers from 1 to 5.


In [11]:
# Check that every language label uses an agreed category.
# Display the distribution without changing any annotations.

allowed_languages = [
    "English", "Malay", "Code-switched", "Other/uncertain"
]

invalid_languages = df[
    ~df["Language Classification"].isin(allowed_languages)
]

print("Records with invalid language labels:", len(invalid_languages))

if not invalid_languages.empty:
    display(
        invalid_languages[
            ["Review ID", "Language Classification"]
        ]
    )

language_counts = (
    df["Language Classification"]
    .value_counts()
    .reindex(allowed_languages, fill_value=0)
)

display(language_counts.rename("Reviews").to_frame())
print("Total:", language_counts.sum())

Records with invalid language labels: 0


,Reviews
Language Classification,
English,89
Malay,33
Code-switched,78
Other/uncertain,0


Total: 200


In [15]:
# Check switching-type labels against each review's language category.
# Allow multiple switching types separated by semicolons.
# This checks annotation consistency without changing any labels.

allowed_types = {
    "Intra-sentential",
    "Inter-sentential",
    "Tag switching"
}

def switching_label_is_valid(row):
    language = row["Language Classification"]
    switching_type = row["Code-Switching Type"]

    if language == "Code-switched":
        types = [
            item.strip()
            for item in switching_type.split(";")
        ]
        return all(item in allowed_types for item in types)

    if language in {"English", "Malay"}:
        return switching_type == "N/A"

    return switching_type in {"N/A", "Uncertain"}

invalid_switching = df[
    ~df.apply(switching_label_is_valid, axis=1)
]

print("Records with inconsistent switching labels:",
      len(invalid_switching))

if not invalid_switching.empty:
    display(
        invalid_switching[
            ["Review ID", "Language Classification",
             "Code-Switching Type"]
        ]
    )
else:
    print("All switching labels match the language categories.")

Records with inconsistent switching labels: 0
All switching labels match the language categories.


In [16]:
# Create a separate working copy for analysis.
# Retain repeated short reviews and the four known source conflicts.
# Save the original text so we can check that it remains unchanged.

df_clean = df.copy()
original_text = df["Review Text"].copy(deep=True)

print("Loaded reviews:", len(df))
print("Reviews retained:", len(df_clean))
print("Reviews removed:", len(df) - len(df_clean))
print(
    "Original text preserved:",
    df_clean["Review Text"].equals(original_text)
)

Loaded reviews: 200
Reviews retained: 200
Reviews removed: 0
Original text preserved: True


In [17]:
# Create minimally standardized text for counting expressions.
# Preserve the original Review Text column and informal spellings.

def prepare_analysis_text(text):
    return (
        text.lower()
        .replace("’", "'")
        .replace("‘", "'")
    )

df_clean["Analysis Text"] = (
    df_clean["Review Text"].apply(prepare_analysis_text)
)

display(
    df_clean[
        ["Review ID", "Review Text", "Analysis Text"]
    ].head(5)
)

print(
    "Original text preserved:",
    df_clean["Review Text"].equals(original_text)
)

,Review ID,Review Text,Analysis Text
0,REV-001,I’m quite disappointed with the food quality. ...,i'm quite disappointed with the food quality. ...
1,REV-002,srsla kubis korng bg mcmni je???? I pay for 13...,srsla kubis korng bg mcmni je???? i pay for 13...
2,REV-003,"Hmmmm judging from the picture, you can say it...","hmmmm judging from the picture, you can say it..."
3,REV-004,I tak suka complain about food. But i bought t...,i tak suka complain about food. but i bought t...
4,REV-005,Foods okay but lousy customer service i feel. ...,foods okay but lousy customer service i feel. ...


Original text preserved: True


In [18]:
# Select reviews labeled Code-switched for the main analysis.
# Use a separate copy so the full dataset remains unchanged.

mixed_df = df_clean[
    df_clean["Language Classification"].eq("Code-switched")
].copy()

print("Full dataset:", len(df_clean))
print("Code-switched subset:", len(mixed_df))

display(
    mixed_df[
        ["Review ID", "Language Classification", "Analysis Text"]
    ].head(5)
)

Full dataset: 200
Code-switched subset: 78


,Review ID,Language Classification,Analysis Text
0,REV-001,Code-switched,i'm quite disappointed with the food quality. ...
1,REV-002,Code-switched,srsla kubis korng bg mcmni je???? i pay for 13...
3,REV-004,Code-switched,i tak suka complain about food. but i bought t...
4,REV-005,Code-switched,foods okay but lousy customer service i feel. ...
6,REV-007,Code-switched,fast services and love the food. maybe boleh b...


In [19]:
# Split analysis text at sentence-ending punctuation and line breaks.
# Keep decimal points inside numbers, such as 13.90.
# This is a practical heuristic, not a complete sentence detector.

import re

boundary_pattern = re.compile(
    r"[!?\r\n…]+|(?<!\d)\.|\.(?!\d)"
)

def split_segments(text):
    return [
        segment.strip()
        for segment in boundary_pattern.split(text)
        if segment.strip()
    ]

# Illustrative example — not a collected review.
example = "ayam sedap. service lambat! harga 13.90… boleh repeat?"

print("Example:", example)
print("Segments:", split_segments(example))

Example: ayam sedap. service lambat! harga 13.90… boleh repeat?
Segments: ['ayam sedap', 'service lambat', 'harga 13.90', 'boleh repeat']


In [21]:
# Extract words, numbers, and decimal prices from one segment.
# Preserve informal spellings, internal apostrophes, and hyphens.
# Exclude punctuation and emojis from tokens without changing original text.

token_pattern = re.compile(
    r"(?:rm)?\d+(?:\.\d+)+|[^\W_]+(?:['-][^\W_]+)*",
    flags=re.UNICODE | re.IGNORECASE
)

def tokenize_segment(segment):
    return token_pattern.findall(segment)

# Illustrative example — not a collected review.
example_segment = "sedappp, ayam dia well-seasoned! it's RM13.90 👍"

print("Tokens:", tokenize_segment(example_segment))

Tokens: ['sedappp', 'ayam', 'dia', 'well-seasoned', "it's", 'RM13.90']


In [22]:
# Split each mixed review into segments, then tokenize each segment.
# Keep segments separate so future n-grams cannot cross their boundaries.
# Store the result in a new column without changing the review text.

mixed_df["Token Segments"] = mixed_df["Analysis Text"].apply(
    lambda text: [
        tokens
        for segment in split_segments(text)
        if (tokens := tokenize_segment(segment))
    ]
)

print("Mixed reviews processed:", len(mixed_df))
print(
    "Reviews with no usable tokens:",
    mixed_df["Token Segments"].apply(len).eq(0).sum()
)

# Inspect the first review's segments.
print("\nFirst review ID:", mixed_df.iloc[0]["Review ID"])

for number, tokens in enumerate(
    mixed_df.iloc[0]["Token Segments"], start=1
):
    print(f"Segment {number}:", tokens)

Mixed reviews processed: 78
Reviews with no usable tokens: 0

First review ID: REV-001
Segment 1: ["i'm", 'quite', 'disappointed', 'with', 'the', 'food', 'quality']
Segment 2: ['the', 'rice', 'had', 'a', 'burnt', 'hangus', 'smell', 'and', 'the', 'chicken', 'was', 'not', 'properly', 'fried']
Segment 3: ['could', 'you', 'please', 'look', 'into', 'this']
